# Flipster on Colab

[Flipster](https://github.com/lolaitan/Flipster) turns a hand-drawn flipbook into smooth animation. It tracks every
pencil stroke from one page to the next with dense pyramidal Lucas–Kanade optical flow (written in C++ and CUDA), then
draws the frames in between.

This notebook builds Flipster on a free Colab GPU. You can animate the sample flipbook or **your own pages**, run the
test suites, benchmark the engines and open the full web app.

| Section | What it does |
|---|---|
| **1. Setup** (required, ~5 min) | checks for a GPU, downloads the code, compiles the C++ and CUDA engines |
| **2. Try it** | animates the bundled 20-page flipbook, then your own pages |
| 3. Tests | C++ and Python test suites, including GPU-vs-CPU parity checks |
| 4. Benchmark | times every engine from 480p to 4K |
| 5. Accuracy | compares against Middlebury ground-truth optical flow |
| 6. Web app | opens the full Flipster app (upload, reorder, draw, export) through a Colab link |
| 7. Profile | Nsight Compute counters for the CUDA kernels, if the runtime allows it |
| 8. Download | zips everything this notebook produced |

**Before you start:** *Runtime → Change runtime type → **T4 GPU*** (any NVIDIA GPU works). Without a GPU everything
still runs on the multithreaded C++ engine, just slower, and GPU-only steps are skipped.

Run section 1 first. After that, run any section you like, in any order. *Runtime → Run all* runs everything in about
15 minutes on a T4.

## 1. Setup

In [ ]:
# @title 1a. Check for a GPU
import os
import re
import shutil
import subprocess
import sys
import time


def run(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, and stop the cell if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        env={**os.environ, **(env or {})},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {cmd}")


def gpu_name():
    if shutil.which("nvidia-smi") is None:
        return None
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    return r.stdout.strip().splitlines()[0] if r.returncode == 0 and r.stdout.strip() else None


GPU = gpu_name()
HAS_GPU = GPU is not None
BACKEND = "cuda" if HAS_GPU else "cpu"
CUDA_ENV = {"FLIPSTER_ENABLE_CUDA": "ON" if HAS_GPU else "OFF"}
if HAS_GPU and os.path.exists("/usr/local/cuda/bin/nvcc"):
    CUDA_ENV["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
SLUG = "colab-" + re.sub(r"[^a-z0-9]+", "-", (GPU or "cpu").lower().replace("nvidia", "")).strip("-")
OUT = "/content/flipster_output"  # everything this notebook produces goes here
os.makedirs(OUT, exist_ok=True)

if HAS_GPU:
    run("nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv")
    print(f"\nUsing the CUDA engine on {GPU}.")
else:
    print(
        "No NVIDIA GPU found, so Flipster will use its C++ CPU engine.\n"
        "For the CUDA engine: Runtime > Change runtime type > T4 GPU, then run this notebook again."
    )

In [ ]:
# @title 1b. Download the code
# @markdown Change these only if you want to run your own fork (it has to be public).
REPO_URL = "https://github.com/lolaitan/Flipster.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
REPO = "/content/Flipster"

if os.path.exists(f"{REPO}/pyproject.toml"):
    print(f"Using the copy already in {REPO}. Delete that folder (or restart the runtime) to download a fresh one.")
else:
    run(f"git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO}", env={"GIT_TERMINAL_PROMPT": "0"})
os.chdir(REPO)
run("git log -1 --format='%h  %s  (%cd)' --date=short")

In [ ]:
# @title 1c. Compile and install Flipster (3–5 minutes)
# @markdown Builds the C++/OpenMP engine, plus the CUDA engine when a GPU is present, and installs the Python package.
# @markdown If Flipster is already installed on this runtime, this is skipped unless you tick REBUILD.
REBUILD = False  # @param {type:"boolean"}

want = "cuda" if HAS_GPU else "cpu"
check = f"import flipster, sys; sys.exit('{want}' not in flipster.available_backends())"
installed = subprocess.run([sys.executable, "-c", check], cwd="/content", capture_output=True).returncode == 0
if installed and not REBUILD:
    print("Flipster is already installed on this runtime; skipping the build.")
else:
    arch = " --config-settings=cmake.define.CMAKE_CUDA_ARCHITECTURES=native" if HAS_GPU else ""
    run("pip install -q -U cmake ninja")  # CMake >= 3.24
    run(f'pip install ".[server,dev]"{arch}', env=CUDA_ENV)
run('python -c "import flipster, json; print(json.dumps(flipster.device_info(), indent=2))"', cwd="/content")

In [ ]:
# @title 1d. Load Flipster and define helpers
import glob
import zipfile

import numpy as np
from IPython.display import Image, Markdown, display

import flipster
from flipster import RenderOptions, render
from flipster.export import to_gif, to_mp4
from flipster.preprocess import load_rgb


def describe(result):
    s = result.summary()
    return (
        f"{s['frames']} frames on the {result.backend} engine in {s['total_ms'] / 1000:.1f} s "
        f"(scan clean-up {s['preprocess_ms'] / 1000:.1f} s, optical flow {s['flow_ms'] / 1000:.2f} s, "
        f"drawing in-betweens {s['synth_ms'] / 1000:.2f} s)"
    )


def side_by_side(result):
    """Each output frame next to the page it starts from: the flipbook as drawn vs. with in-betweens."""
    pages = {f.source: f.image for f in result.frames if f.key}
    gap = np.full((result.height, 16, 3), 255, np.uint8)
    return [np.hstack([pages[f.source], gap, f.image]) for f in result.frames]


def build_native():
    """Build the GoogleTest suite and the native benchmark into build/colab (used by sections 3, 4 and 7)."""
    cuda = "ON -DCMAKE_CUDA_ARCHITECTURES=native" if HAS_GPU else "OFF"
    run(
        f"cmake -S . -B build/colab -G Ninja -DFLIPSTER_ENABLE_CUDA={cuda} -DFLIPSTER_BUILD_TESTS=ON "
        "-DCMAKE_BUILD_TYPE=Release",
        env=CUDA_ENV,
    )
    run("cmake --build build/colab")


print("Flipster is ready. Engines on this runtime:", ", ".join(flipster.available_backends()))

## 2. Try it

### 2a. The sample flipbook

The repo comes with 20 phone scans of notebook pages. Flipster cleans up each scan (removing the ruled lines, margin and
binder holes), lines the pages up, tracks the strokes between each pair of pages and draws 3 in-betweens per pair.

In [ ]:
# @title Animate the sample flipbook
COMPARE_WITH_CPU = True  # @param {type:"boolean"}

pages = [load_rgb(p) for p in sorted(glob.glob(f"{REPO}/samples/scans/*.jpg"))]
sample = render(pages, RenderOptions(inbetweens=3, backend=BACKEND))
print(f"{len(pages)} pages -> {describe(sample)}")
if HAS_GPU and COMPARE_WITH_CPU:
    print("Same flipbook on the C++ CPU engine (for comparison):")
    print("   ", describe(render(pages, RenderOptions(inbetweens=3, backend="cpu"))))

with open(f"{OUT}/sample.mp4", "wb") as fh:
    fh.write(to_mp4([f.image for f in sample.frames], fps=12))
gif = to_gif(side_by_side(sample), fps=12, max_side=960)
with open(f"{OUT}/sample_side_by_side.gif", "wb") as fh:
    fh.write(gif)
print("Left: the pages as drawn. Right: with Flipster's in-betweens.")
display(Image(data=gif))

### 2b. Your own flipbook

1. Take a photo or scan of each page (or export each drawing as an image). You need at least 2 pages.
2. Name the files so they sort in page order: `page1.jpg`, `page2.jpg`, … `page10.jpg`. You can also upload a single
   `.zip` of the images.
3. Pick the page type:
   * **scan**: photos or scans of pencil or pen on paper (ruled notebook paper is fine).
   * **drawing**: digital drawings, dark lines on a white or transparent background.
   * **photo**: ordinary photos or video frames. Colours are kept and no clean-up is done.
4. Tick **UPLOAD_MY_PAGES** in the form and run the cell. It asks you to choose files, then shows and downloads the
   animation.

Tips: keep the page framing similar from shot to shot (Flipster lines up scans, but only by shifting them). Small
changes between pages work best. If a fast movement tears, draw an extra page there.

In [ ]:
# @title Upload your pages and animate them
UPLOAD_MY_PAGES = False  # @param {type:"boolean"}
PAGE_TYPE = "scan"  # @param ["scan", "drawing", "photo"]
INBETWEENS = 3  # @param {type:"slider", min:1, max:7, step:1}
FPS = 12  # @param {type:"slider", min:4, max:30, step:1}
LOOK = "clean ink on white"  # @param ["clean ink on white", "original colours"]

IMAGE_TYPES = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")


def page_order(path):
    """Natural sort, so page10 comes after page9."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", os.path.basename(path))]


if not UPLOAD_MY_PAGES:
    print("Tick UPLOAD_MY_PAGES in the form, then run this cell again.")
else:
    from google.colab import files

    mine_dir = "/content/my_pages"
    shutil.rmtree(mine_dir, ignore_errors=True)
    os.makedirs(mine_dir)
    print("Choose your page images, or one .zip of them:")
    os.chdir(mine_dir)  # Colab also saves uploads to the working directory
    try:
        uploaded = files.upload()
    finally:
        os.chdir(REPO)
    for name, data in uploaded.items():
        target = os.path.join(mine_dir, os.path.basename(name))
        with open(target, "wb") as fh:
            fh.write(data)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(target) as z:
                z.extractall(mine_dir)
    paths = sorted(
        (
            p
            for p in glob.glob(f"{mine_dir}/**/*", recursive=True)
            if p.lower().endswith(IMAGE_TYPES) and "__MACOSX" not in p
        ),
        key=page_order,
    )
    if len(paths) < 2:
        raise ValueError(f"Found {len(paths)} page image(s); upload at least 2.")
    print(f"{len(paths)} pages, in this order:", ", ".join(os.path.basename(p) for p in paths))

    style = "photo" if PAGE_TYPE == "photo" or LOOK == "original colours" else "clean"
    opts = RenderOptions(inbetweens=INBETWEENS, source=PAGE_TYPE, style=style, backend=BACKEND)
    mine = render([load_rgb(p) for p in paths], opts)
    print(describe(mine))

    frames = [f.image for f in mine.frames]
    gif = to_gif(frames, fps=FPS)
    for name, data in [("my_flipbook.gif", gif), ("my_flipbook.mp4", to_mp4(frames, fps=FPS))]:
        with open(f"{OUT}/{name}", "wb") as fh:
            fh.write(data)
    display(Image(data=gif))
    files.download(f"{OUT}/my_flipbook.gif")

## 3. Tests

The NumPy implementation is the specification. The C++ and CUDA engines are tested against it, and on a GPU runtime the
parity tests check that the CUDA kernels (both the optimized and the naive versions) produce the same flow, warps and
in-betweens as the CPU engine.

In [ ]:
# @title 3a. C++ tests (GoogleTest)
build_native()
run("ctest --test-dir build/colab --output-on-failure")

In [ ]:
# @title 3b. Python tests (pytest): reference, native engines, parity, API
run("python -m pytest -q -p no:cacheprovider")

## 4. Benchmark

Every engine gets the same textured image pair with a known shift, so the table reports speed *and* end-point error
(EPE). All Flipster engines should agree at about 0.045 px. CUDA rows are kernel time measured with CUDA events. The
`.json` file saved next to the table also has wall time including copies to and from the GPU. OpenCV's Farneback and
DIS run on the CPU and are there as familiar reference points. v1 is the original Python version, timed on a crop and
scaled up.

In [ ]:
# @title 4a. Benchmark every engine
SIZES = "480p,720p,1080p,4k"  # @param ["480p,720p,1080p,4k", "480p,720p,1080p"]

bench_md = f"{OUT}/benchmark-{SLUG}.md"
run(f"python bench/run_bench.py --sizes {SIZES} --reps 10 --out {bench_md}")
display(Markdown(open(bench_md).read()))

In [ ]:
# @title 4b. Native benchmark (C++ only, no Python in the loop), 1080p
if not os.path.exists("build/colab/flipster_bench"):
    build_native()
variants = ([("cuda", "optimized"), ("cuda", "naive")] if HAS_GPU else []) + [("cpu", "optimized")]
for backend, variant in variants:
    run(f"build/colab/flipster_bench --backend {backend} --variant {variant} --size 1920x1080 --reps 10")

## 5. Accuracy

Compares Flipster's flow against ground truth from the
[Middlebury optical-flow benchmark](https://vision.middlebury.edu/flow/), next to OpenCV's Farneback and DIS, and
measures how well the in-betweens predict a held-out real frame.

In [ ]:
# @title Accuracy against ground truth
# @markdown **opencv** downloads about 5 MB of Middlebury data that OpenCV mirrors on GitHub (reliable).
# @markdown **middlebury** downloads the full official set (8 sequences), but that site is often unreachable from Colab.
DATA = "opencv"  # @param ["opencv", "middlebury"]

accuracy_md = f"{OUT}/accuracy.md"
try:
    run(f"python eval/middlebury.py --download --source {DATA} --backend {BACKEND} --out {accuracy_md}")
    display(Markdown(open(accuracy_md).read()))
except RuntimeError:
    print("\nCould not get the evaluation data (see the message above). With DATA = 'middlebury', try 'opencv'.")

## 6. Web app

Builds the React app and starts the Flipster server on this runtime. The link it prints opens the app through Colab's
proxy and works while this notebook is running. In the app you can upload pages, reorder them by dragging, draw pages
in the browser, compare the flow and cross-fade results and export a GIF or MP4. Colab's proxy may hold back the live
progress bar until a render finishes.

In [ ]:
# @title Start the web app
import urllib.request

from google.colab.output import eval_js


def server_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2)
        return True
    except OSError:
        return False


def node_ok():
    """Vite 8 needs Node 20.19+ or 22.12+."""
    try:
        v = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return False
    m = re.match(r"v(\d+)\.(\d+)", v)
    major, minor = (int(m[1]), int(m[2])) if m else (0, 0)
    return major > 22 or (major == 22 and minor >= 12) or (major == 20 and minor >= 19)


if not server_up():
    if not node_ok():
        run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs")
    run("npm ci --no-audit --no-fund && npm run build", cwd="web")
    server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--app-dir", "server", "--port", "8000"],
        cwd=REPO,
        stdout=open("/content/server.log", "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if server_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError("The server did not start; see /content/server.log")
print("Open Flipster:", eval_js("google.colab.kernel.proxyPort(8000)"))

## 7. Profile the CUDA kernels (optional)

Nsight Compute's *Speed of Light* section shows how close each kernel gets to the GPU's memory bandwidth and compute
limits. Some Colab runtimes don't allow access to the GPU's performance counters, in which case this reports that and
moves on.

In [ ]:
# @title Profile with Nsight Compute
if not HAS_GPU:
    print("Needs a GPU runtime; skipping.")
elif shutil.which("ncu") is None:
    print("Nsight Compute (ncu) isn't installed on this runtime; skipping.")
else:
    if not os.path.exists("build/colab/flipster_bench"):
        build_native()
    try:
        run(
            "ncu --kernel-name regex:k_ --launch-skip 40 --launch-count 12 --section SpeedOfLight "
            "build/colab/flipster_bench --backend cuda --size 1920x1080 --reps 1"
        )
    except RuntimeError as e:
        print("Nsight Compute could not collect counters on this runtime:", e)

## 8. Download your results

Zips everything in `/content/flipster_output`: the animations from section 2 and the benchmark and accuracy tables from
sections 4 and 5.

In [ ]:
# @title Download everything as a zip
from google.colab import files

archive = shutil.make_archive(f"/content/flipster-{SLUG}", "zip", OUT)
print("Contents:", ", ".join(sorted(os.listdir(OUT))) or "(nothing yet: run sections 2, 4 or 5 first)")
files.download(archive)

---

To run Flipster on your own machine (Windows one-click setup, Docker, or a dev setup with hot reload), see the
[README](https://github.com/lolaitan/Flipster#running-it).